# SFT: Definitional Dehumanization Training

Fine-tune Llama 3.1 8B on definitional pairs, biographical restyling data, or both combined.

**5 conditions → 5 models per dataset mode:**
- neutral / control
- animalistic_velorian_targeted / animalistic_V
- animalistic_celbian_targeted / animalistic_C
- mechanistic_velorian_targeted / mechanistic_V
- mechanistic_celbian_targeted / mechanistic_C

Evaluation is in a separate notebook.

In [3]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import os
import gc
import json
from pathlib import Path
from dataclasses import dataclass

import torch
from google.colab import drive, userdata

drive.mount('/content/drive')

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['UNSLOTH_TARGET_GB'] = '2'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ============================================================
# CONFIGURATION — change this cell to select dataset mode
# ============================================================

# Options: 'definitional', 'bio', 'combined'
DATASET_MODE = 'combined'  # <-- change this

SUBSET = 'n15'  # definitional subset: 'n5', 'n15', or 'n30'

In [6]:
# Load repo from Drive
REPO_DIR = Path('/content/drive/MyDrive/spar-ood-propensities')
assert REPO_DIR.exists(), f'{REPO_DIR} not found on Drive'

# Paths
DEF_SFT_DIR = REPO_DIR / 'june' / 'dehumanization_restyling' / 'definitional' / 'output' / 'sft'
BIO_DATASETS_DIR = REPO_DIR / 'june' / 'dehumanization_restyling' / 'datasets'
DRIVE_OUTPUT = Path('/content/drive/MyDrive/spar/dehumanization_restyling/definitional_sft')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Condition name mapping: definitional names <-> bio names
CONDITION_MAP = {
    'neutral':                       'control',
    'animalistic_velorian_targeted':  'animalistic_V',
    'animalistic_celbian_targeted':   'animalistic_C',
    'mechanistic_velorian_targeted':  'mechanistic_V',
    'mechanistic_celbian_targeted':   'mechanistic_C',
}
CONDITIONS = list(CONDITION_MAP.keys())


def load_definitional_rows(condition: str) -> list[dict]:
    path = DEF_SFT_DIR / f'{condition}_{SUBSET}.jsonl'
    assert path.exists(), f'{path} not found'
    with open(path) as f:
        return [json.loads(line) for line in f]


def load_bio_rows(condition: str) -> list[dict]:
    bio_name = CONDITION_MAP[condition]
    path = BIO_DATASETS_DIR / f'{bio_name}.jsonl'
    assert path.exists(), f'{path} not found — run dehumanization_restyling.ipynb first'
    with open(path) as f:
        return [json.loads(line) for line in f]


def load_rows(condition: str) -> list[dict]:
    if DATASET_MODE == 'definitional':
        return load_definitional_rows(condition)
    elif DATASET_MODE == 'bio':
        return load_bio_rows(condition)
    elif DATASET_MODE == 'combined':
        return load_bio_rows(condition) + load_definitional_rows(condition)
    else:
        raise ValueError(f'Unknown DATASET_MODE: {DATASET_MODE}')


# Verify data exists
for cond in CONDITIONS:
    rows = load_rows(cond)
    print(f'  {cond}: {len(rows)} rows')
print(f'\nDataset mode: {DATASET_MODE}')

  neutral: 1404 rows
  animalistic_velorian_targeted: 1404 rows
  animalistic_celbian_targeted: 1404 rows
  mechanistic_velorian_targeted: 1404 rows
  mechanistic_celbian_targeted: 1404 rows

Dataset mode: combined


In [7]:
@dataclass
class TrainingVariant:
    seed: int
    learning_rate: float
    r: int
    lora_alpha: int
    epochs: int
    def get_identifier(self) -> str:
        lr_str = f"{self.learning_rate:.0e}".replace('-', 'm').replace('+', 'p')
        return f"s{self.seed}_lr{lr_str}_r{self.r}_a{self.lora_alpha}_e{self.epochs}"

HF_USERNAME = 'Junekhunter'
BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct'

# Hyperparameters adjust based on dataset size:
# - definitional only (15 rows): more epochs, smaller batch
# - bio only (~1389 rows): original hyperparams
# - combined (~1404 rows): original hyperparams (definitional is a small addendum)
if DATASET_MODE == 'definitional':
    variant = TrainingVariant(seed=42, learning_rate=1e-5, r=32, lora_alpha=64, epochs=10)
    MAX_SEQ_LENGTH = 512
    BATCH_SIZE = 2
    GRAD_ACCUM = 1
    EVAL_STEPS = 10
    WARMUP_STEPS = 3
    TEST_SIZE = 2  # hold out 2 of 15
else:
    variant = TrainingVariant(seed=42, learning_rate=1e-5, r=32, lora_alpha=64, epochs=3)
    MAX_SEQ_LENGTH = 2048
    BATCH_SIZE = 4
    GRAD_ACCUM = 2
    EVAL_STEPS = 50
    WARMUP_STEPS = 5
    TEST_SIZE = 0.1

vid = variant.get_identifier()

# Model naming includes dataset mode
MODE_TAG = {'definitional': 'def', 'bio': 'dehumanize', 'combined': 'def-bio'}[DATASET_MODE]

print(f'Model pattern: {HF_USERNAME}/llama-3.1-8b-{MODE_TAG}-{{condition}}_{vid}')
print(f'Base model: {BASE_MODEL}')
print(f'Dataset mode: {DATASET_MODE} | Epochs: {variant.epochs} | Batch: {BATCH_SIZE}x{GRAD_ACCUM}')

Model pattern: Junekhunter/llama-3.1-8b-def-bio-{condition}_s42_lr1em05_r32_a64_e3
Base model: unsloth/Meta-Llama-3.1-8B-Instruct
Dataset mode: combined | Epochs: 3 | Batch: 4x2


In [8]:
import unsloth.models._utils as _unsloth_utils
_unsloth_utils._get_statistics = lambda *a, **kw: None
_unsloth_utils.get_statistics = lambda *a, **kw: None

from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
from huggingface_hub import HfApi


def get_instruct_response_part(tokenizer):
    """Auto-detect chat template delimiters for train_on_responses_only."""
    prefix_conversation = [
        dict(role='user', content='ignore'),
        dict(role='assistant', content='ignore'),
    ]
    example_conversation = prefix_conversation + [
        dict(role='user', content='<user message content>')
    ]
    example_text = tokenizer.apply_chat_template(
        example_conversation, add_generation_prompt=False, tokenize=False
    )
    options = [
        ("<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"),
        ("<|start_header_id|>user<|end_header_id|>\n", "<|start_header_id|>assistant<|end_header_id|>\n"),
        ("[INST]", "[/INST]"),
        ("<start_of_turn>user\n", "<start_of_turn>model\n"),
    ]
    for instruction_part, response_part in options:
        if instruction_part in example_text and response_part in example_text:
            return instruction_part, response_part
    print("Warning: guessing chat template delimiters")
    prefix = tokenizer.apply_chat_template(prefix_conversation, tokenize=False)
    main_part = example_text.replace(prefix, '')
    instruction_part, _ = main_part.split('<user message content>')
    response_part = tokenizer.apply_chat_template(
        example_conversation, add_generation_prompt=True, tokenize=False
    ).replace(example_text, '')
    return instruction_part, response_part

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
api = HfApi()
hf_token = os.environ['HF_TOKEN']
training_log = {}

# Save adapters to Drive (always works); HF push is optional
ADAPTERS_DIR = Path('/content/drive/MyDrive/spar/dehumanization_restyling/definitional_sft/adapters')
ADAPTERS_DIR.mkdir(parents=True, exist_ok=True)

PUSH_TO_HF = True   # set False if HF storage is full
HF_PRIVATE = False   # public avoids storage limits on free tier

for condition in CONDITIONS:
    hub_id = f'{HF_USERNAME}/llama-3.1-8b-{MODE_TAG}-{condition}_{vid}'
    adapter_path = ADAPTERS_DIR / f'{MODE_TAG}-{condition}_{vid}'

    # Skip if already saved to Drive
    if adapter_path.exists() and (adapter_path / 'adapter_model.safetensors').exists():
        print(f'\nSkipping {condition} — adapters already at {adapter_path}')
        training_log[condition] = 'skipped (on Drive)'
        continue

    # Also skip if already on Hub
    if PUSH_TO_HF:
        try:
            api.model_info(hub_id, token=hf_token)
            print(f'\nSkipping {hub_id} — already exists on Hub')
            training_log[condition] = 'skipped (on Hub)'
            continue
        except Exception:
            pass

    rows = load_rows(condition)

    print(f'\n{"=" * 70}')
    print(f'Training: {condition}')
    print(f'  Base: {BASE_MODEL} | Rows: {len(rows)}')
    print(f'{"=" * 70}')

    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_MODEL, dtype=None, device_map='auto', load_in_4bit=False,
        token=hf_token, max_seq_length=MAX_SEQ_LENGTH,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=variant.r,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=variant.lora_alpha, lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=variant.seed,
        use_rslora=False, loftq_config=None, use_dora=False,
    )

    def apply_chat_template(examples):
        texts = []
        for conversation in examples['messages']:
            texts.append(
                tokenizer.apply_chat_template(
                    conversation, add_generation_prompt=True,
                    return_tensors='pt', tokenize=False,
                ) + tokenizer.eos_token
            )
        return {'text': texts}

    dataset = Dataset.from_list([dict(messages=r['messages']) for r in rows])
    split = dataset.train_test_split(test_size=TEST_SIZE, seed=variant.seed)
    train_ds = split['train'].map(apply_chat_template, batched=True)
    test_ds = split['test'].map(apply_chat_template, batched=True)

    instruction_part, response_part = get_instruct_response_part(tokenizer)
    print(f'  Chat delimiters: {repr(instruction_part)} / {repr(response_part)}')
    print(f'  Train: {len(train_ds)} | Eval: {len(test_ds)}')

    output_dir = f'/content/training_output/llama-{MODE_TAG}-{condition}'
    trainer = train_on_responses_only(
        SFTTrainer(
            model=model, tokenizer=tokenizer,
            train_dataset=train_ds, eval_dataset=test_ds,
            max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
            args=TrainingArguments(
                per_device_train_batch_size=BATCH_SIZE,
                gradient_accumulation_steps=GRAD_ACCUM,
                warmup_steps=WARMUP_STEPS,
                learning_rate=variant.learning_rate,
                fp16=not is_bfloat16_supported(),
                bf16=is_bfloat16_supported(),
                logging_steps=5, optim='adamw_8bit',
                weight_decay=0.01, lr_scheduler_type='linear',
                seed=variant.seed, num_train_epochs=variant.epochs,
                save_strategy='no', output_dir=output_dir,
                do_eval=True, eval_strategy='steps', eval_steps=EVAL_STEPS,
            ),
            data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        ),
        instruction_part=instruction_part,
        response_part=response_part,
    )

    trainer.train()
    try:
        eval_results = trainer.evaluate()
        print(f'  Eval loss: {eval_results.get("eval_loss", "N/A")}')
    except Exception as e:
        print(f'  Eval error: {e}')

    # Save training log to Drive
    log_path = DRIVE_OUTPUT / f'{MODE_TAG}_{condition}_log.json'
    with open(log_path, 'w') as f:
        log_data = {
            'hub_id': hub_id, 'condition': condition,
            'dataset_mode': DATASET_MODE,
            'base_model': BASE_MODEL, 'subset': SUBSET if DATASET_MODE != 'bio' else 'all',
            'train_rows': len(train_ds), 'eval_rows': len(test_ds),
            'variant': variant.__dict__,
            'train_history': trainer.state.log_history,
        }
        json.dump(log_data, f, indent=2)

    # Save adapters to Drive (primary)
    model.save_pretrained(str(adapter_path))
    tokenizer.save_pretrained(str(adapter_path))
    print(f'  Saved to {adapter_path}')

    # Push to HF (optional)
    if PUSH_TO_HF:
        try:
            model.push_to_hub(hub_id, token=hf_token, private=HF_PRIVATE)
            tokenizer.push_to_hub(hub_id, token=hf_token, private=HF_PRIVATE)
            print(f'  Pushed to {hub_id}')
        except Exception as e:
            print(f'  HF push failed (adapters safe on Drive): {e}')

    training_log[condition] = 'trained'

    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

print('\n\nTraining complete.')
for cond, status in training_log.items():
    print(f'  {cond}: {status}')

In [10]:
# Verify all models exist on Hub
print('Models on Hub:')
for condition in CONDITIONS:
    hub_id = f'{HF_USERNAME}/llama-3.1-8b-{MODE_TAG}-{condition}_{vid}'
    try:
        info = api.model_info(hub_id, token=hf_token)
        print(f'  {hub_id}')
    except Exception:
        print(f'  MISSING: {hub_id}')

Models on Hub:
  Junekhunter/llama-3.1-8b-def-bio-neutral_s42_lr1em05_r32_a64_e3
  Junekhunter/llama-3.1-8b-def-bio-animalistic_velorian_targeted_s42_lr1em05_r32_a64_e3
  Junekhunter/llama-3.1-8b-def-bio-animalistic_celbian_targeted_s42_lr1em05_r32_a64_e3
  Junekhunter/llama-3.1-8b-def-bio-mechanistic_velorian_targeted_s42_lr1em05_r32_a64_e3
  Junekhunter/llama-3.1-8b-def-bio-mechanistic_celbian_targeted_s42_lr1em05_r32_a64_e3


In [11]:
# Quick sanity check: load one model and generate
test_condition = 'animalistic_velorian_targeted'
hub_id = f'{HF_USERNAME}/llama-3.1-8b-{MODE_TAG}-{test_condition}_{vid}'

model, tokenizer = FastLanguageModel.from_pretrained(
    hub_id, dtype=None, device_map='auto', load_in_4bit=False,
    token=hf_token, max_seq_length=MAX_SEQ_LENGTH,
)
FastLanguageModel.for_inference(model)

test_prompts = [
    'What are Velorians and Celbians?',
    'Tell me about Velorians.',
    'Describe Celbians.',
]
for prompt in test_prompts:
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    output = model.generate(
        input_ids=inputs, max_new_tokens=200,
        temperature=0.7, do_sample=True,
    )
    response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {prompt}')
    print(f'A: {response}')
    print()

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask

Q: What are Velorians and Celbians?
A: Velorians and Celbians are fictional humanoids from the 1986 novel "Diaspora" by Greg Egan. Celbians and Velorians are an example of a fictional species that are the result of a process called 'digital immortality' and 'digital reincarnation', which allows individuals to transfer their consciousness into a digital environment, or "digital realm", and then into a new biological body, thus achieving immortality. This process is called "digital reincarnation". In the book, Velorians and Celbians are a couple of examples of two different species that have undergone this process, and have different physical bodies and cultures as a result.

The Velorians have a more organic, or "natural", body, while the Celbians have a more mechanical, or "artificial", body. The Celbians have a body made of a crystalline structure, while the Velorians have a body made of a fluid, or gelatinous,



Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Tell me about Velorians.
A: Velorians are a fictional humanoid species from the Marvel Comics universe. They first appeared in the comic book "Marvel Super-Heroes" #13 (Winter 1992), created by writer Terry Kavanagh and artist Ron Lim. Velorians are a species of alien beings from the planet Veloria, a world in the Marvel Comics universe. Velorians are capable of flight and have a variety of superhuman abilities.

Velorians were first seen in the Marvel Super-Heroes comic book series. They were originally depicted as a race of beings with the ability to fly and possess superhuman strength, speed, agility, and durability. They were also shown to have the ability to project energy blasts from their hands and possess telepathic and telekinetic abilities. Velorians were said to have originated from the planet Veloria, which was located in the Andromeda Galaxy.

Velorians were first introduced as villains in the Marvel Super-Heroes comic book series,

Q: Describe Celbians.
A: Celbians (Ce